# Bounding-box object insertion with the Easy-Insert LoRA (FLUX.2 [klein] base 4B)

Reference-based insertion: **background + bbox + reference image of the object → object placed in the box**.
No training: this loads the pretrained community LoRA [`LiXiY/Easy-Insert`](https://huggingface.co/LiXiY/Easy-Insert) on top of `FLUX.2-klein-base-4B`.

> ### ⚠️ One thing I could not verify
> The model card only says the LoRA takes *"a background image with a masked region and a reference image"*. I could not read the author's inference code (<https://github.com/huan-yin/Easy-Insert>), so **I don't know the exact input format it was trained on** (how the masked region is drawn, image order, prompt wording).
>
> So this notebook makes the input format a switch (`packing=`), ships a few plausible formats, and includes **`compare_packings(...)`** which runs them side by side on your image so you can see which one the LoRA actually responds to. **Before trusting results, open the repo's README / inference script and set `packing` and `PROMPT` to match it.**

Pipeline: crop around bbox → pack inputs → best-of-N → change-aware blend region (keeps overflow) → Poisson/alpha blend. Pixels outside the blend region stay bit-identical to the input.

In [ ]:
import torch
from diffusers import Flux2KleinPipeline
from huggingface_hub import list_repo_files

device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_ID = "black-forest-labs/FLUX.2-klein-base-4B"   # what the LoRA targets (base, not distilled)
LORA_REPO = "LiXiY/Easy-Insert"
LORA_FILE = None                                       # set if the repo has several .safetensors files

pipe = Flux2KleinPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16)

files = [f for f in list_repo_files(LORA_REPO) if f.endswith(".safetensors")]
print("LoRA files in repo:", files)
LORA_FILE = LORA_FILE or (files[0] if len(files) == 1 else None)
if LORA_FILE is None:
    raise SystemExit("Multiple/no .safetensors files found: set LORA_FILE to the right one (see README).")

pipe.load_lora_weights(LORA_REPO, weight_name=LORA_FILE, adapter_name="insert")
pipe.set_adapters(["insert"], adapter_weights=[1.0])   # LoRA strength; try 0.8-1.25

pipe.to(device)                        # needs ~13+ GB VRAM
# pipe.enable_model_cpu_offload()      # use this instead if you run out of memory

# If load_lora_weights raises a KeyError about 'single_blocks...' it's a diffusers LoRA-conversion issue:
# make sure diffusers is installed from git main (cell above) and restart the runtime.

## Helpers

In [ ]:
import io
import numpy as np
import cv2
from scipy import ndimage as ndi
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import torch

# ======================================================================
# INPUT PACKING  <-- the part to match with the Easy-Insert repo's README
# ======================================================================
PROMPT = ("Insert the object from the reference image into the masked region of the background image. "
          "Keep the background unchanged and match the lighting and perspective.")

# name -> (fill color of the masked box on the background, also send a separate mask image?)
PACKINGS = {
    "magenta_box": dict(fill=(255, 0, 255), mask_image=False),   # [background(magenta box), reference]
    "black_box":   dict(fill=(0, 0, 0),     mask_image=False),   # [background(black box),   reference]
    "gray_box":    dict(fill=(128, 128, 128), mask_image=False), # [background(gray box),    reference]
    "mask_image":  dict(fill=(0, 0, 0),     mask_image=True),    # [background(black box), white-on-black mask, reference]
}


def build_inputs(work, wbox, reference, packing, ref_max_side=768):
    """Return (list_of_images, fill_color) for the pipeline."""
    cfg = PACKINGS[packing]
    bg = work.copy()
    ImageDraw.Draw(bg).rectangle(wbox, fill=cfg["fill"])
    ref = reference.convert("RGB")
    if max(ref.size) > ref_max_side:
        s = ref_max_side / max(ref.size)
        ref = ref.resize((round(ref.width * s), round(ref.height * s)), Image.LANCZOS)
    imgs = [bg]
    if cfg["mask_image"]:
        m = Image.new("RGB", work.size, (0, 0, 0))
        ImageDraw.Draw(m).rectangle(wbox, fill=(255, 255, 255))
        imgs.append(m)
    imgs.append(ref)
    return imgs, cfg["fill"]


# ---------- geometry ----------
def get_crop_box(bbox, img_size, context_scale=2.5, min_side=256):
    x0, y0, x1, y1 = bbox
    W, H = img_size
    w, h = x1 - x0, y1 - y0
    cw = min(W, max(w * context_scale, min_side))
    ch = min(H, max(h * context_scale, min_side))
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    left = min(max(cx - cw / 2, 0), W - cw)
    top = min(max(cy - ch / 2, 0), H - ch)
    return tuple(int(round(v)) for v in (left, top, left + cw, top + ch))


def work_size(w, h, target):
    s = target / max(w, h)
    return max(16, round(w * s / 16) * 16), max(16, round(h * s / 16) * 16)


def auto_target(cw, ch, lo=512, hi=1024):
    return int(min(hi, max(lo, max(cw, ch))))


def box_mask_np(shape_hw, box):
    h, w = shape_hw
    m = np.zeros((h, w), bool)
    x0, y0, x1, y1 = [int(round(v)) for v in box]
    m[max(y0, 0):min(y1, h), max(x0, 0):min(x1, w)] = True
    return m


# ---------- candidate analysis ----------
def analyze_candidate(out, work, wbox, fill, thr=28):
    o = np.asarray(out, np.int16)
    w = np.asarray(work, np.int16)
    diff = np.abs(o - w).max(2).astype(np.float32)
    diff = cv2.GaussianBlur(diff, (0, 0), 2)
    changed = ndi.binary_opening(diff > thr, iterations=2)
    bm = box_mask_np(diff.shape, wbox)
    area = max(bm.sum(), 1)
    overflow = (changed & ~bm).sum() / area
    inside_change = (changed & bm).sum() / area
    # leftover placeholder color inside the box (skip for black/gray: too common in real content)
    leftover = 0.0
    if fill == (255, 0, 255):
        leftover = ((np.abs(o - np.array(fill)).sum(2) < 90) & bm).sum() / area
    score = overflow + 3.0 * leftover + (1.0 if inside_change < 0.05 else 0.0)
    return dict(score=float(score), overflow=float(overflow), inside=float(inside_change),
                leftover=float(leftover), changed=changed, box=bm)


def build_region(changed, bm, grow=6, max_growth=3.0):
    region = changed | bm
    lab, _ = ndi.label(region)
    keep = np.unique(lab[bm]); keep = keep[keep > 0]
    region = np.isin(lab, keep)
    if region.sum() > max_growth * bm.sum():
        region = bm.copy()
    return ndi.binary_dilation(region, iterations=grow)


# ---------- blending / matching ----------
def poisson_blend(src, dst, region):
    pad = 32
    s = cv2.copyMakeBorder(src, pad, pad, pad, pad, cv2.BORDER_REFLECT)
    d = cv2.copyMakeBorder(dst, pad, pad, pad, pad, cv2.BORDER_REFLECT)
    m = cv2.copyMakeBorder(region.astype(np.uint8) * 255, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=0)
    ys, xs = np.where(m > 0)
    center = (int((xs.min() + xs.max()) // 2), int((ys.min() + ys.max()) // 2))
    res = cv2.seamlessClone(s, d, m, center, cv2.NORMAL_CLONE)
    return res[pad:-pad, pad:-pad]


def _lap_var(gray, mask):
    return float(cv2.Laplacian(gray, cv2.CV_32F)[mask].var())


def match_sharpness(gen, ref, region, max_blur=2.0):
    ring = ndi.binary_dilation(region, iterations=40) & ~ndi.binary_dilation(region, iterations=4)
    if ring.sum() < 100:
        return gen, 0.0
    g_ref = cv2.cvtColor(ref, cv2.COLOR_RGB2GRAY)
    g_gen = cv2.cvtColor(gen, cv2.COLOR_RGB2GRAY)
    target = np.log(_lap_var(g_ref, ring) + 1e-6)
    sigmas = [s for s in (0, 0.5, 0.8, 1.2, 1.6, 2.0, 2.5, 3.0) if s <= max_blur]
    def err(s):
        g = cv2.GaussianBlur(g_gen, (0, 0), s) if s > 0 else g_gen
        return abs(np.log(_lap_var(g, region) + 1e-6) - target)
    best = min(sigmas, key=err)
    return (cv2.GaussianBlur(gen, (0, 0), best) if best > 0 else gen), best


def jpeg_roundtrip(arr, q):
    buf = io.BytesIO()
    Image.fromarray(arr).save(buf, "JPEG", quality=q)
    return np.asarray(Image.open(buf).convert("RGB"))


# ---------- main ----------
@torch.no_grad()
def insert_object_lora(
    pipe, image, bbox, reference,
    packing="magenta_box", prompt=None,
    n_candidates=3, seed=0,
    context_scale=2.5, target=None,
    steps=50, guidance=4.0,                 # base-model settings
    thr=28, grow=6, feather=8, max_growth=3.0,
    blend="poisson",                        # "poisson" or "alpha"
    match_quality=False, max_blur=2.0, jpeg_quality=None,
    ref_max_side=768,
):
    """
    image : PIL image;  bbox : (x0, y0, x1, y1) pixels;  reference : PIL image of the object to insert.
    Returns (final_image, debug_dict)
    """
    image = image.convert("RGB")
    x0, y0, x1, y1 = bbox
    assert 0 <= x0 < x1 <= image.width and 0 <= y0 < y1 <= image.height, "bbox outside image"
    prompt = prompt or PROMPT

    # 1) crop with context near native resolution
    crop_box = get_crop_box(bbox, image.size, context_scale)
    crop = image.crop(crop_box)
    cw, ch = crop.size
    ww, wh = work_size(cw, ch, target or auto_target(cw, ch))
    sx, sy = ww / cw, wh / ch
    wbox = ((x0 - crop_box[0]) * sx, (y0 - crop_box[1]) * sy,
            (x1 - crop_box[0]) * sx, (y1 - crop_box[1]) * sy)
    work = crop.resize((ww, wh), Image.LANCZOS)

    # 2) pack inputs the way the LoRA expects (see PACKINGS)
    inputs, fill = build_inputs(work, wbox, reference, packing, ref_max_side)

    # 3) best-of-N
    cands = []
    gdev = pipe.device if hasattr(pipe, "device") else "cpu"
    for i in range(n_candidates):
        gen = torch.Generator(device=gdev).manual_seed(seed + i)
        out = pipe(image=inputs, prompt=prompt, height=wh, width=ww,
                   num_inference_steps=steps, guidance_scale=guidance, generator=gen).images[0].convert("RGB")
        if out.size != (ww, wh):
            out = out.resize((ww, wh), Image.LANCZOS)
        a = analyze_candidate(out, work, wbox, fill, thr)
        a.update(seed=seed + i, out=out)
        cands.append(a)
    best = min(cands, key=lambda c: c["score"])
    out = best["out"]

    # 4) change-aware region at full crop resolution
    region_w = build_region(best["changed"], best["box"], grow, max_growth)
    region = cv2.resize(region_w.astype(np.uint8), (cw, ch), interpolation=cv2.INTER_NEAREST).astype(bool)
    gen_c = np.asarray(out.resize((cw, ch), Image.LANCZOS))
    crop_np = np.asarray(crop)

    blur_used = 0.0
    if match_quality:
        gen_c, blur_used = match_sharpness(gen_c, crop_np, region, max_blur)
    if jpeg_quality:
        gen_c = jpeg_roundtrip(gen_c, jpeg_quality)

    # 5) blend
    if blend == "poisson":
        try:
            new_crop = poisson_blend(gen_c, crop_np, region)
        except Exception as e:
            print("poisson failed, falling back to alpha:", e)
            blend = "alpha"
    if blend == "alpha":
        alpha = cv2.GaussianBlur(region.astype(np.float32), (0, 0), max(feather / 2, 0.1))[..., None]
        new_crop = (gen_c * alpha + crop_np * (1 - alpha)).round().astype(np.uint8)

    new_crop = np.where(ndi.binary_dilation(region, iterations=feather + 2)[..., None], new_crop, crop_np)
    final = image.copy()
    final.paste(Image.fromarray(new_crop.astype(np.uint8)), crop_box[:2])

    dbg = dict(prompt=prompt, packing=packing, crop_box=crop_box, inputs=inputs, raw_output=out, region=region,
               candidates=[{k: c[k] for k in ("seed", "score", "overflow", "inside", "leftover", "out")} for c in cands],
               chosen_seed=best["seed"], blur_used=blur_used)
    return final, dbg


# ---------- diagnostics: which input format does the LoRA respond to? ----------
def compare_packings(pipe, image, bbox, reference, packings=None, seed=0, **kw):
    """Run ONE candidate per packing and show raw outputs side by side."""
    packings = packings or list(PACKINGS)
    results = {}
    for p in packings:
        try:
            results[p] = insert_object_lora(pipe, image, bbox, reference, packing=p, n_candidates=1, seed=seed, **kw)
        except Exception as e:
            print(f"[{p}] failed: {e}")
    names = list(results)
    fig, axs = plt.subplots(2, len(names), figsize=(4 * len(names), 7), squeeze=False)
    for j, n in enumerate(names):
        final, d = results[n]
        axs[0, j].imshow(d["raw_output"]); axs[0, j].set_title(f"{n}: raw output", fontsize=10)
        axs[1, j].imshow(final);          axs[1, j].set_title(f"{n}: final", fontsize=10)
        axs[0, j].axis("off"); axs[1, j].axis("off")
    plt.tight_layout(); plt.show()
    return results


# ---------- display ----------
def show(images, titles=None, figsize=(16, 6)):
    fig, axs = plt.subplots(1, len(images), figsize=figsize)
    axs = [axs] if len(images) == 1 else axs
    for i, (ax, im) in enumerate(zip(axs, images)):
        ax.imshow(im); ax.axis("off")
        if titles: ax.set_title(titles[i])
    plt.tight_layout(); plt.show()


def show_candidates(dbg, cols=6):
    cs = dbg["candidates"]
    cols = min(cols, len(cs)); rows = (len(cs) + cols - 1) // cols
    fig, axs = plt.subplots(rows, cols, figsize=(3.2 * cols, 2.6 * rows), squeeze=False)
    for ax in axs.ravel(): ax.axis("off")
    for ax, c in zip(axs.ravel(), cs):
        ax.imshow(c["out"])
        chosen = c["seed"] == dbg["chosen_seed"]
        ax.set_title(f"seed {c['seed']}  score {c['score']:.2f}" + ("  ✓" if chosen else ""),
                     fontsize=9, color="green" if chosen else "black")
    plt.tight_layout(); plt.show()


def preview_bbox(image, bbox, color="red", width=4):
    im = image.convert("RGB").copy()
    ImageDraw.Draw(im).rectangle(bbox, outline=color, width=width)
    return im


def norm_to_px(bbox_norm, size):
    W, H = size
    return (bbox_norm[0] * W, bbox_norm[1] * H, bbox_norm[2] * W, bbox_norm[3] * H)

## Load your background, bbox and reference object

`bbox` is `(x0, y0, x1, y1)` in **pixels** (use `norm_to_px` for normalized boxes). The reference should show the object you want inserted, ideally cropped tight.

In [ ]:
image = Image.open("scene.jpg").convert("RGB")        # <-- background
reference = Image.open("object.png").convert("RGB")   # <-- the object to insert
bbox = (400, 300, 700, 600)                           # <-- where to put it
# bbox = norm_to_px((0.4, 0.5, 0.6, 0.8), image.size)

show([preview_bbox(image, bbox), reference], ["Background + bbox", "Reference"], figsize=(12, 6))

## Step 1 — find the input format the LoRA responds to

Runs one sample per packing. Pick the one where the object appears inside the box and looks like your reference. If none works, read the Easy-Insert repo's inference code and add its exact format to `PACKINGS` / `PROMPT`.

In [ ]:
results = compare_packings(pipe, image, bbox, reference, seed=0)

## Step 2 — full run with the best packing

In [ ]:
PACKING = "magenta_box"     # <-- set to the one that worked above

result, dbg = insert_object_lora(
    pipe, image, bbox, reference,
    packing=PACKING,
    n_candidates=3,          # base model = 50 steps/run, so keep N small
    seed=0,
)

print("Prompt:", dbg["prompt"])
print(f"chosen seed {dbg['chosen_seed']}")
show([image, dbg["inputs"][0], dbg["raw_output"], result],
     ["Original", "Model input (crop)", "Chosen raw output (crop)", "Final"])
show_candidates(dbg)
result.save("result_easy_insert.png")

## Notes & tuning

| Symptom | Try |
|---|---|
| Nothing inserted / box just filled with background | wrong `packing` or `PROMPT`; run `compare_packings`, then check the repo README |
| Magenta/black box visible in the output | use another `packing`, or change the fill color in `PACKINGS` |
| Object doesn't resemble the reference | tighter crop of the reference; raise LoRA strength (`set_adapters(..., adapter_weights=[1.25])`) |
| Object clipped or bigger than the box | overflow is kept by design; tighten the box, raise `n_candidates` |
| Too slow | lower `steps` (try 28), or load the LoRA on the distilled model (`FLUX.2-klein-4B`, `steps=4, guidance=1.0`); quality may drop since the LoRA targets the *base* model |
| Seam / color tint | `blend="alpha"` vs `"poisson"`; raise `grow` |
| Looks too crisp vs. low-quality source | `match_quality=True`, `jpeg_quality=60` |
| `KeyError` on `load_lora_weights` | install diffusers from git main, restart runtime (known Flux2 LoRA-conversion issue) |